
# Notebook 3: Missing Value Handling

## Objective
Understand why missing values occur, how to detect them, and how
to select appropriate techniques for handling them.

## Learning Outcomes
- Understand MCAR, MAR, and MNAR.
- Detect missing values and calculate missing percentages.
- Compare dropping and imputation techniques.
- Apply mean, median, mode, constant, forward-fill, backward-fill,
  and interpolation methods.
- Understand group-based, KNN, and iterative imputation.
- Evaluate the advantages and limitations of each technique.


In [1]:
import pandas as pd
import numpy as np

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Libraries imported successfully!")

Pandas version: 3.0.3
NumPy version: 2.4.6
Libraries imported successfully!



## 1. What Are Missing Values?

Missing values occur when a data field has no recorded value.

In Pandas, missing data is commonly represented by NaN, None,
or NaT for datetime values.

Missing values can affect calculations, visualizations, statistical
analysis, and machine learning models.

They must be investigated before choosing a suitable treatment.



## 2. Why Missing Values Occur

Common causes include:
- Users skipping optional form fields.
- Sensors or devices failing to record measurements.
- Data collection or transmission errors.
- Different systems storing different fields.
- Records being lost during data integration.
- Values being intentionally withheld.
- Data not being applicable to certain records.

The reason for missingness helps determine the most appropriate
handling method.



## 3. Types of Missing Data

### MCAR — Missing Completely At Random
The probability of a value being missing is unrelated to observed
or unobserved data.

Example: A recording system randomly loses a few measurements.

### MAR — Missing At Random
Missingness depends on other observed variables, but not on the
missing value itself after accounting for those variables.

Example: Survey responses are more often missing for a particular
observed age group.

### MNAR — Missing Not At Random
Missingness depends on the missing value itself or other unobserved
information.

Example: People with very high expenses are less likely to report
their expenses.

These mechanisms are conceptual assumptions. A missing-value
pattern alone generally cannot prove which mechanism applies.


In [2]:
data = {
    "Age": [25, 30, np.nan, 40, 35, 28, np.nan, 45],
    "Salary": [30000, 45000, 50000, np.nan, 40000, 35000, 60000, np.nan],
    "Department": [
        "IT", "HR", "IT", np.nan, "Finance", "HR", "Finance", "IT"
    ],
    "Experience": [2, 5, 7, 10, np.nan, 3, 12, 8],
    "Score": [70, 85, 90, 88, 76, np.nan, 95, 82]
}

df_missing = pd.DataFrame(data)

print("Sample dataset:")
display(df_missing)

Sample dataset:


,Age,Salary,Department,Experience,Score
0,25.0,30000.0,IT,2.0,70.0
1,30.0,45000.0,HR,5.0,85.0
2,NaN,50000.0,IT,7.0,90.0
3,40.0,NaN,NaN,10.0,88.0
4,35.0,40000.0,Finance,NaN,76.0
5,28.0,35000.0,HR,3.0,NaN
6,NaN,60000.0,Finance,12.0,95.0
7,45.0,NaN,IT,8.0,82.0


In [3]:
print("Missing values per column:")
display(df_missing.isnull().sum())

print("\nTotal missing values:")
print(df_missing.isnull().sum().sum())

Missing values per column:


Age           2
Salary        2
Department    1
Experience    1
Score         1
dtype: int64


Total missing values:
7


In [4]:
missing_summary = pd.DataFrame({
    "Missing_Count": df_missing.isnull().sum(),
    "Missing_Percentage": (
        df_missing.isnull().mean() * 100
    ).round(2)
})

display(
    missing_summary.sort_values(
        by="Missing_Percentage",
        ascending=False
    )
)

,Missing_Count,Missing_Percentage
Age,2,25.0
Salary,2,25.0
Department,1,12.5
Experience,1,12.5
Score,1,12.5



## 4. Detecting and Measuring Missing Values

`isnull()` and `isna()` identify missing values in a DataFrame.

`sum()` counts missing values in each column, while `mean()` on
a Boolean missing-value mask calculates the missing proportion.

Missing percentages help prioritize columns for investigation.
However, a high percentage alone is not sufficient reason to
delete a column; its importance and the reason for missingness
must also be considered.


In [5]:
rows_dropped = df_missing.dropna()

print("Original shape:", df_missing.shape)
print("Shape after dropping rows with any missing value:", rows_dropped.shape)

display(rows_dropped)

Original shape: (8, 5)
Shape after dropping rows with any missing value: (2, 5)


,Age,Salary,Department,Experience,Score
0,25.0,30000.0,IT,2.0,70.0
1,30.0,45000.0,HR,5.0,85.0


In [6]:
column_threshold = 60  # Demonstration threshold: 60% missing

missing_pct = df_missing.isnull().mean() * 100

columns_to_drop = missing_pct[
    missing_pct > column_threshold
].index.tolist()

df_drop_columns = df_missing.drop(columns=columns_to_drop)

print("Columns selected for dropping:", columns_to_drop)
print("Resulting shape:", df_drop_columns.shape)
display(df_drop_columns.head())

Columns selected for dropping: []
Resulting shape: (8, 5)


,Age,Salary,Department,Experience,Score
0,25.0,30000.0,IT,2.0,70.0
1,30.0,45000.0,HR,5.0,85.0
2,NaN,50000.0,IT,7.0,90.0
3,40.0,NaN,NaN,10.0,88.0
4,35.0,40000.0,Finance,NaN,76.0


In [7]:
column_threshold = 60  # Demonstration threshold: 60% missing

missing_pct = df_missing.isnull().mean() * 100

columns_to_drop = missing_pct[
    missing_pct > column_threshold
].index.tolist()

df_drop_columns = df_missing.drop(columns=columns_to_drop)

print("Columns selected for dropping:", columns_to_drop)
print("Resulting shape:", df_drop_columns.shape)
display(df_drop_columns.head())

Columns selected for dropping: []
Resulting shape: (8, 5)


,Age,Salary,Department,Experience,Score
0,25.0,30000.0,IT,2.0,70.0
1,30.0,45000.0,HR,5.0,85.0
2,NaN,50000.0,IT,7.0,90.0
3,40.0,NaN,NaN,10.0,88.0
4,35.0,40000.0,Finance,NaN,76.0


In [8]:
df_mean = df_missing.copy()

df_mean["Age"] = df_mean["Age"].fillna(
    df_mean["Age"].mean()
)

print("Age after mean imputation:")
display(df_mean[["Age"]])

Age after mean imputation:


,Age
0,25.000000
1,30.000000
2,33.833333
3,40.000000
4,35.000000
5,28.000000
6,33.833333
7,45.000000


In [9]:
df_median = df_missing.copy()

df_median["Salary"] = df_median["Salary"].fillna(
    df_median["Salary"].median()
)

print("Salary after median imputation:")
display(df_median[["Salary"]])

Salary after median imputation:


,Salary
0,30000.0
1,45000.0
2,50000.0
3,42500.0
4,40000.0
5,35000.0
6,60000.0
7,42500.0


In [10]:
df_mode = df_missing.copy()

mode_value = df_mode["Department"].mode().iloc[0]

df_mode["Department"] = df_mode["Department"].fillna(mode_value)

print("Mode used:", mode_value)
display(df_mode[["Department"]])

Mode used: IT


,Department
0,IT
1,HR
2,IT
3,IT
4,Finance
5,HR
6,Finance
7,IT


In [11]:
df_constant = df_missing.copy()

df_constant["Department"] = (
    df_constant["Department"].fillna("Unknown")
)

print("Department after constant imputation:")
display(df_constant[["Department"]])

Department after constant imputation:


,Department
0,IT
1,HR
2,IT
3,Unknown
4,Finance
5,HR
6,Finance
7,IT


In [12]:
time_data = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=6, freq="D"),
    "Temperature": [25, np.nan, 27, np.nan, 29, 30]
})

time_data["Forward_Fill"] = time_data["Temperature"].ffill()

display(time_data)

,Date,Temperature,Forward_Fill
0,2026-01-01,25.0,25.0
1,2026-01-02,NaN,25.0
2,2026-01-03,27.0,27.0
3,2026-01-04,NaN,27.0
4,2026-01-05,29.0,29.0
5,2026-01-06,30.0,30.0


In [13]:
time_data["Backward_Fill"] = time_data["Temperature"].bfill()

display(time_data)

,Date,Temperature,Forward_Fill,Backward_Fill
0,2026-01-01,25.0,25.0,25.0
1,2026-01-02,NaN,25.0,27.0
2,2026-01-03,27.0,27.0,27.0
3,2026-01-04,NaN,27.0,29.0
4,2026-01-05,29.0,29.0,29.0
5,2026-01-06,30.0,30.0,30.0


In [14]:
interpolation_data = pd.DataFrame({
    "Measurement": [10.0, np.nan, 20.0, np.nan, 30.0]
})

interpolation_data["Interpolated"] = (
    interpolation_data["Measurement"].interpolate(method="linear")
)

display(interpolation_data)

,Measurement,Interpolated
0,10.0,10.0
1,NaN,15.0
2,20.0,20.0
3,NaN,25.0
4,30.0,30.0


In [15]:
group_data = pd.DataFrame({
    "Department": ["IT", "IT", "HR", "HR", "Finance", "Finance"],
    "Salary": [50000, np.nan, 35000, np.nan, 60000, 65000]
})

group_data["Salary"] = group_data["Salary"].fillna(
    group_data.groupby("Department")["Salary"].transform("median")
)

display(group_data)

,Department,Salary
0,IT,50000.0
1,IT,50000.0
2,HR,35000.0
3,HR,35000.0
4,Finance,60000.0
5,Finance,65000.0


In [16]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

knn_data = df_missing[["Age", "Salary", "Experience", "Score"]].copy()

# Scale features so large-valued columns do not dominate distance.
scaler = StandardScaler()
scaled_values = scaler.fit_transform(knn_data)

knn_imputer = KNNImputer(n_neighbors=3)

knn_imputed_scaled = knn_imputer.fit_transform(scaled_values)

knn_imputed = pd.DataFrame(
    scaler.inverse_transform(knn_imputed_scaled),
    columns=knn_data.columns
)

display(knn_imputed)

,Age,Salary,Experience,Score
0,25.000000,30000.0,2.000000,70.0
1,30.000000,45000.0,5.000000,85.0
2,38.333333,50000.0,7.000000,90.0
3,40.000000,50000.0,10.000000,88.0
4,35.000000,40000.0,3.333333,76.0
5,28.000000,35000.0,3.000000,77.0
6,38.333333,60000.0,12.000000,95.0
7,45.000000,45000.0,8.000000,82.0


In [17]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

iterative_data = df_missing[["Age", "Salary", "Experience", "Score"]].copy()

iterative_imputer = IterativeImputer(
    max_iter=10,
    random_state=42
)

iterative_result = iterative_imputer.fit_transform(iterative_data)

iterative_df = pd.DataFrame(
    iterative_result,
    columns=iterative_data.columns
)

display(iterative_df)

,Age,Salary,Experience,Score
0,25.000000,30000.000000,2.000000,70.00000
1,30.000000,45000.000000,5.000000,85.00000
2,39.820041,50000.000000,7.000000,90.00000
3,40.000000,52907.068208,10.000000,88.00000
4,35.000000,40000.000000,6.300457,76.00000
5,28.000000,35000.000000,3.000000,74.04361
6,46.788008,60000.000000,12.000000,95.00000
7,45.000000,45550.148324,8.000000,82.00000



## Comparison of Missing-Value Handling Techniques

| Technique | Best suited for | Main limitation |
|---|---|---|
| Drop rows | Few incomplete records | Information loss |
| Drop columns | Low-value features with excessive missingness | Loss of useful features |
| Mean | Roughly symmetric numerical data | Sensitive to outliers |
| Median | Skewed numerical data | Reduces variability |
| Mode | Categorical data | Overrepresents common categories |
| Constant | Meaningful default or unknown category | Can distort distributions |
| Forward fill | Ordered observations | Stale values |
| Backward fill | Ordered observations | Potential future-data leakage |
| Interpolation | Ordered continuous measurements | Assumes a pattern |
| Group-based | Meaningful subgroups | Unstable small groups |
| KNN | Similar records with informative features | Distance and scaling sensitivity |
| Iterative | Relationships among numerical features | Model assumptions and computational cost |


In [18]:
print("Missing values in KNN-imputed dataset:")
display(knn_imputed.isnull().sum())

print("\nMissing values in iterative-imputed dataset:")
display(iterative_df.isnull().sum())

Missing values in KNN-imputed dataset:


Age           0
Salary        0
Experience    0
Score         0
dtype: int64


Missing values in iterative-imputed dataset:


Age           0
Salary        0
Experience    0
Score         0
dtype: int64


## Conclusion

In this notebook, I learned why missing values occur and studied
MCAR, MAR, and MNAR missingness mechanisms.

I practiced detecting missing values, calculating missing
percentages, dropping incomplete records or columns, and applying
mean, median, mode, and constant-value imputation.

I also explored forward fill, backward fill, interpolation,
group-based imputation, KNN imputation, and iterative imputation.

Each technique has advantages and limitations. The correct method
depends on the data distribution, the reason values are missing,
the feature's meaning, and the intended machine learning task.

Missing values should never be filled blindly with the mean.
Preprocessing decisions should be documented, validated, and
performed without leaking information from validation or test data.
